# 01 - Exploratory data analysis

**Owner:** Person 1 (shared with everyone)


Understand the ISUP distribution, the provider split and the patch counts before
any model is trained. ISUP imbalance is the single biggest reason accuracy is a
misleading metric for this task -- QWK is the headline number instead.

Run after `data/common_split.csv` exists (or point `MANIFEST` at the mock
manifest to see the notebook work today).


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

In [ ]:
MANIFEST = ROOT / "data/mock/manifest.csv" if (ROOT / "data/mock/manifest.csv").exists() else ROOT / "data/common_split.csv"
FEATURES = MANIFEST.parent / "features"
print(MANIFEST)

## Split and class distribution

In [ ]:
from src.dataset import class_distribution, describe_dataset, read_manifest

summary = describe_dataset(MANIFEST, FEATURES)
print(json.dumps({k: v for k, v in summary.items() if k != "class_distribution"}, indent=2))

records = read_manifest(MANIFEST)
df = pd.DataFrame(class_distribution(records)).T.sort_index()
df.index = df.index.astype(int)
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.6))
df.plot(kind="bar", ax=ax)
ax.set_xlabel("ISUP grade group"); ax.set_ylabel("slides"); ax.set_title("Class distribution per split")
ax.legend(frameon=False)
plt.tight_layout()

## Provider breakdown (a known PANDA confounder)

In [ ]:
providers = pd.Series([r.data_provider for r in records]).value_counts()
print(providers)
print("\nProvider x grade:")
pd.crosstab(pd.Series([r.data_provider for r in records]), pd.Series([r.label for r in records]))

## Patch counts per slide (memory planning)

In [ ]:
from src.dataset import feature_path, load_feature_bag

counts = []
for record in records[:40]:
    try:
        features, coords, _ = load_feature_bag(feature_path(FEATURES, record.image_id))
        counts.append({"slide_id": record.image_id, "n_patches": features.size(0), "has_coords": coords is not None})
    except FileNotFoundError:
        pass
pd.DataFrame(counts).describe(include="all") if counts else print("No feature files found yet.")

## Takeaways to carry into the report

* Which grades are scarce (usually 0 and 5) -> justifies class weights and balanced accuracy.
* Whether both data providers appear in every split -> a distribution shift worth mentioning.
* Typical patch count -> sets `max_patches` in the training config.